In [0]:
from pyspark.sql.functions import col, from_unixtime, to_date, trim, length, upper

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("bronze_schema", "valeriimatviiv_bronze", "2. Bronze Schema")
dbutils.widgets.text("silver_schema", "valeriimatviiv_silver", "3. Silver Schema")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

bronze_table = f"{catalog}.{bronze_schema}.finnhub_news_bronze"
silver_table = f"{catalog}.{silver_schema}.finnhub_news_silver"

In [0]:
df_bronze = spark.read.table(bronze_table)

# Parse Unix epoch timestamp, format news text, and deduplicate
df_silver = (
    df_bronze
    .withColumn("ArticleId", col("id").cast("string"))
    .withColumn("Symbol", upper(trim(col("related"))))
    .withColumn("NewsTimestamp", from_unixtime(col("datetime").cast("long")).cast("timestamp"))
    .withColumn("NewsDate", to_date(from_unixtime(col("datetime").cast("long"))))
    .withColumn("Headline", trim(col("headline")))
    .withColumn("Summary", trim(col("summary")))
    .filter(col("Headline").isNotNull() & (length(col("Headline")) > 0))
    .filter(col("NewsTimestamp").isNotNull())
    .dropDuplicates(["ArticleId"])
    .select(
        col("ArticleId"),
        col("Symbol"),
        col("NewsTimestamp"),
        col("NewsDate"),
        col("Headline"),
        col("Summary"),
        col("url"),
        col("source"),
        col("_schema_phase")
    )
)

# Overwrite Silver Delta Table
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

# print(f"Successfully processed {df_silver.count()} clean news articles into Silver table: {silver_table}")

In [0]:
# df_verify_news = spark.table(f"{catalog}.{silver_schema}.finnhub_news_silver")

# print(f"Total Silver News Articles: {df_verify_news.count()}")
# print("Schema:")
# df_verify_news.printSchema()
# display(df_verify_news.limit(5))